# HydraY NNUE: 1024 -> 16 -> 32 -> 1 on the v10 pool

Runtime -> **G4 (NVIDIA RTX PRO 6000)**, set `ARM` in the cell "configuration", then
"Run all". About **75 GiB of free disk**: the pool is 63 GB.

The net of `colab_v10.ipynb` with a **second hidden layer of 32** after l1
(`HYDRAY_L2=32` in `trainer_deep.rs`, branch `nnue-l2x32`). Everything else is the
recipe of the shipped net: same v10 pool, eval scale 356, 8 output buckets, same LR
schedules, QAT on l0 and l1. The new layer and the output layer stay float in the
file, as the 16 -> 1 tail always has.

Speed, measured 2026-11-12 on the laptop with nets of this shape whose extra weights
are zero (identical tree): -2% nps on an E-core with the layer in int8, about -4% in
float (`audit-2026-11-10/reports/W-width/README.md`).

### Arms

| `ARM` | phases | what it answers |
|---|---|---|
| `pre` | `antidead` 160 sb from scratch | the new shape with the pre-training phase only |
| `ft` | `ft-shipped-qat` 40 sb from the **shipped net** | the fine-tune only: l0 and l1 come from the shipped net, l2 (16 -> 32) and l3 (32 -> 1) start fresh |
| `both` | `antidead` 160 sb, then `ft-antidead-qat` 40 sb from it | the full recipe of the shipped net on the new shape |

| phase | superbatches | recipe |
|---|---|---|
| `antidead` | 160 | StepLR 1e-3, x0.1 at superbatch 80; l1 and l2 biases start at 0.1, warmup 3000 batches, float tail unclipped |
| `ft-antidead-qat` | 40 | from the `antidead` net: cosine 1e-4 -> 2.5e-6, warmup 3000, dead l1 neurons revived, QAT |
| `ft-shipped-qat` | 40 | the same, from `nnue/net/hydray.nnue` |

`pre` and `both` share the `antidead` phase: a phase whose final net is on Drive is
skipped, so running `pre` first and `both` afterwards trains it once.

**`ft` caveat**: the new layers start from random weights (He init, fixed seed, l2 bias
0.1) and get only the fine-tune's small LR (1e-4 down to 2.5e-6): the arm measures
whether 40 superbatches are enough for them, not only the shape. The start check
(5 positions, bullet's forward against `sanity_deep.rs`) still runs, on the random head.

**Data**: `v10/` of the Hugging Face dataset `ThomasGhione/hydray`, 1,954,186,654
positions, the pool of the shipped net, downloaded in 16 parts and checked against the
sha256 of `v10/manifest.json`, plus the holdout `hydray_v10.holdout.bin`.

Nets on Drive: `hydray-v10-l2x32-<phase>-<superbatches>sb.nnue`, checkpoints every
`STAGE` superbatches. The engine cannot load this layout yet (see the last cell).


In [ ]:
# --- helper: any failed command stops the notebook, output streams live ---
import subprocess, os, re, glob, hashlib

def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end='', flush=True)
        out.append(line)
    if p.wait() != 0 and check:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')
    return ''.join(out)

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
sh('nproc; free -g | head -2; df -h /content | tail -1')

In [ ]:
# --- configuration ---
import json, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

ARM    = 'both'          # pre | ft | both  (see the table above)
STAGE  = 20              # checkpoint to Drive every STAGE superbatches (multiple of 10)
BRANCH = 'nnue-l2x32'    # trainer_deep.rs with HYDRAY_L2
BUCKETS = 8              # the 4.2.0 layout
EVAL_SCALE = '356'       # 400 / 1.124: v9's targets at the recipe's 400 (label unit, see above)
DOWNLOADS  = 8           # parts downloaded at once

REPO    = '/content/hydray'
TRAINER = f'{REPO}/nnue/trainer'
LOCAL   = '/content/data'
INIT_NET = f'{REPO}/nnue/net/hydray.nnue'          # the shipped net (v10 ft-antidead-qat), checked by sha256 below
INIT_SHA = '844d3ca82c3c4cf41e354bee866726e45ef5ac8ab2cd2c1786fe0a6cc8f4fa83'
HF_V10   = 'https://huggingface.co/datasets/ThomasGhione/hydray/resolve/main/v10'
V10_RECORDS = 1_954_186_654

L2 = '32'                # width of the second hidden layer
ANTIDEAD = {'HYDRAY_L1B_INIT': '0.1', 'HYDRAY_WARMUP': '3000', 'HYDRAY_FLOAT_CLIP': '64'}
FT = {'HYDRAY_LR': 'cosine', 'HYDRAY_LR_START': '0.0001', 'HYDRAY_LR_FINAL': '0.0000025',
      'HYDRAY_WARMUP': '3000', 'HYDRAY_FLOAT_CLIP': '64', 'HYDRAY_REVIVE': '1', 'HYDRAY_L1B_INIT': '0.1'}
ARMS = {  # name: phases run in order, each (id, superbatches, environment of the trainer)
    'pre':  [('antidead',        160, ANTIDEAD)],
    'ft':   [('ft-shipped-qat',  40,  dict(FT, HYDRAY_QAT='1', HYDRAY_INIT_NET=INIT_NET))],
    'both': [('antidead',        160, ANTIDEAD),
             ('ft-antidead-qat', 40,  dict(FT, HYDRAY_QAT='1', HYDRAY_INIT_NET=f'{DRIVE}/hydray-v10-l2x32-antidead-160sb.nnue'))],
}
assert ARM in ARMS, f'unknown ARM {ARM}'
PHASES = [(p, sb, dict(env, HYDRAY_L2=L2)) for p, sb, env in ARMS[ARM]]
if EVAL_SCALE != '400':
    PHASES = [(p, sb, dict(env, HYDRAY_EVAL_SCALE=EVAL_SCALE)) for p, sb, env in PHASES]

def use_phase(i):
    """Makes phase i current: the stage helpers read these globals."""
    global PHASE, TOTAL_SB, ENV, NET_ID, ENV_PREFIX
    PHASE, TOTAL_SB, ENV = PHASES[i]
    NET_ID = f'hydray-v10-l2x32-{PHASE}-{TOTAL_SB}sb'
    ENV_PREFIX = ' '.join(f'{k}={v}' for k, v in ENV.items())

assert gpu, 'no GPU: change the runtime type to G4 (RTX PRO 6000)'
if 'RTX PRO 6000' not in gpu:
    print(f'WARNING: {gpu} is not the RTX PRO 6000 this notebook is set up for')
assert STAGE % 10 == 0 and all(sb % STAGE == 0 for _, sb, _ in PHASES), \
    'stage ends must be checkpoints (save_rate 10)'

NET_FILE_BYTES = 6_443_584      # 8 buckets with the 16 -> 32 -> 1 tail (sanity_deep.rs)
SHIPPED_BYTES  = 6_425_664      # the shipped layout, the only one l1census reads
TRAINER_LINE   = f'trainer_deep: 1024 -> 16 -> {L2} -> 1, {BUCKETS} output buckets'

# The manifest of the pool the shipped net trained on (v10 files unchanged on Hugging Face
# since 2026-11-09 10:44 UTC, before the 4.4.0 release): any other manifest stops the run.
V10_MANIFEST_SHA = '0d88118b0a1a48b351a35aa389f6e34d895ad695622db10aa8c921ff0d94334c'
_raw = urllib.request.urlopen(f'{HF_V10}/manifest.json').read()
assert hashlib.sha256(_raw).hexdigest() == V10_MANIFEST_SHA, 'v10/manifest.json on Hugging Face has changed: not the pool of the shipped net'
MANIFEST = json.loads(_raw)
assert MANIFEST['records'] == V10_RECORDS and not MANIFEST['test_bytes'], 'not the v10 manifest'
print(f'arm {ARM}')
print(f"v10: {MANIFEST['records']:,} records in {len(MANIFEST['parts'])} parts, "
      f"holdout {MANIFEST['holdout']['records']:,}")
for i in range(len(PHASES)):
    use_phase(i)
    print(f'  phase {PHASE}: {TOTAL_SB} superbatches, net id {NET_ID}, environment {ENV}')

In [ ]:
# CUDA toolkit for this GPU: NVRTC compiles bullet's kernels for sm_<compute capability>,
# and a Blackwell GPU (12.x) needs CUDA >= 12.8. An older toolkit gets the 12.8 libraries
# next to it (NVIDIA's apt repository), and the build and the trainer use those.
import json
cc = subprocess.run('nvidia-smi --query-gpu=compute_cap --format=csv,noheader', shell=True,
                    capture_output=True, text=True).stdout.split()[0]
def toolkit(path):
    try:
        v = json.load(open(f'{path}/version.json'))['cuda']['version']
    except (OSError, KeyError, ValueError):
        out = subprocess.run(f'{path}/bin/nvcc --version', shell=True, capture_output=True, text=True).stdout
        m = re.search(r'release (\d+)\.(\d+)', out)
        v = f'{m.group(1)}.{m.group(2)}' if m else '0.0'
    return tuple(int(x) for x in v.split('.')[:2])
NEED = (12, 8) if int(cc.split('.')[0]) >= 12 else (11, 0)
CUDA = '/usr/local/cuda'
print(f'compute capability {cc}, toolkit {toolkit(CUDA)} at {CUDA}, needed {NEED}')
if toolkit(CUDA) < NEED:
    distro = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True,
                            capture_output=True, text=True, executable='/bin/bash').stdout.strip()
    sh(f'cd /tmp && wget -q https://developer.download.nvidia.com/compute/cuda/repos/{distro}/x86_64/cuda-keyring_1.1-1_all.deb'
       ' && dpkg -i cuda-keyring_1.1-1_all.deb >/dev/null && apt-get -qq update'
       ' && apt-get -qq install -y cuda-nvrtc-dev-12-8 cuda-cudart-dev-12-8 libcublas-dev-12-8 cuda-driver-dev-12-8 >/dev/null')
    CUDA = '/usr/local/cuda-12.8'
    # bullet's build.rs links from CUDA_PATH/lib64.
    if not os.path.exists(f'{CUDA}/lib64'):
        os.symlink(f'{CUDA}/targets/x86_64-linux/lib', f'{CUDA}/lib64')
    assert all(os.path.exists(f'{CUDA}/lib64/lib{l}.so') for l in ('nvrtc', 'cudart', 'cublas')), \
        f'the CUDA 12.8 libraries are not in {CUDA}/lib64'
print('CUDA_PATH', CUDA)

# --- Rust, clone, checks, build ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh(f'rm -rf {REPO} && git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY {REPO}')
sh(f'cd {REPO} && git log --oneline -1')

# Not ceremonial: a past run trained for an hour an architecture other than the one
# believed, because the clone was wrong and a checkpoint name says nothing about its contents.
tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr and 'const L1_SIZE: usize = 16;' in tr
assert 'HYDRAY_INIT_NET' in tr and 'HYDRAY_FLOAT_CLIP' in tr, 'trainer_deep.rs has no recipe knobs: wrong branch'
assert 'HYDRAY_QAT' in tr and 'PAIRWISE_NUDGE' in sd, 'trainer_deep.rs has no QAT: wrong branch'
assert 'pub fn to_bullet' in sd and 'pub fn dead_rows' in sd
assert 'HYDRAY_L2' in tr and 'pub const L2_SIZE: usize = 32;' in sd, 'no second hidden layer: wrong branch (nnue-l2x32)'
assert hashlib.sha256(open(INIT_NET, 'rb').read()).hexdigest() == INIT_SHA, 'nnue/net/hydray.nnue is not the shipped net (v10 ft-antidead-qat)'

CARGO = f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH env CUDA_PATH={CUDA} LD_LIBRARY_PATH={CUDA}/lib64:$LD_LIBRARY_PATH HYDRAY_OUTPUT_BUCKETS={BUCKETS}'
sh(f'{CARGO} cargo build -r --bin trainer_deep --features cuda')
sh(f'{CARGO} cargo build -r --bin sanity_deep')
sh(f'{TRAINER}/target/release/sanity_deep {INIT_NET} --init-check')

# The activation census (C++, the engine's own NNUE code). If this compiler cannot
# build it, the census falls back to the all-zero rows that sanity_deep counts.
L1CENSUS = '/content/l1census'
for std in ('c++23', 'c++2b'):
    sh(f'cd {REPO} && g++ -std={std} -O2 -march=native nnue/tools/l1census.cpp nnue/nnue_deep.cpp -o {L1CENSUS}', check=False)
    if os.path.exists(L1CENSUS):
        break
print('l1census:', 'built' if os.path.exists(L1CENSUS) else 'NOT built, zero rows only')

In [ ]:
# --- data: the 16 parts of v10 and its holdout, in parallel, each checked by sha256 ---
# A part already on local disk with the right size and checked sha256 (a restart in the
# same session) is not downloaded again. A broken download resumes where it stopped.
os.makedirs(LOCAL, exist_ok=True)
WANT = [(p['file'], p['bytes'], p['sha256']) for p in MANIFEST['parts']]
HOLD = MANIFEST['holdout']
WANT.append((HOLD['file'], 32 * HOLD['records'], HOLD['sha256']))

need = sum(size for _, size, _ in WANT)
have = sum(os.path.getsize(f'{LOCAL}/{n}') for n, _, _ in WANT if os.path.exists(f'{LOCAL}/{n}'))
st = os.statvfs('/content'); free = st.f_bavail * st.f_frsize
print(f'free {free/2**30:.1f} GiB, still to download {(need - have)/2**30:.1f} GiB')
assert free > (need - have) + 8 * 2**30, 'not enough disk: use an A100 runtime (or one with more disk)'

def fetch(item):
    name, size, sha = item
    path, ok = f'{LOCAL}/{name}', f'{LOCAL}/{name}.sha256ok'
    if os.path.exists(path) and os.path.getsize(path) == size \
            and os.path.exists(ok) and open(ok).read() == sha:
        return f'{name}: already here'
    t0 = time.time()
    for attempt in range(10):
        if os.path.exists(path) and os.path.getsize(path) > size:
            os.remove(path)
        if not (os.path.exists(path) and os.path.getsize(path) == size):
            subprocess.run(['curl', '-sfL', '--retry', '5', '-C', '-', '-o', path, f'{HF_V10}/{name}'])
        if os.path.exists(path) and os.path.getsize(path) == size:
            break
        time.sleep(10 * (attempt + 1))
    got = os.path.getsize(path) if os.path.exists(path) else 0
    assert got == size, f'{name}: {got} bytes after 10 attempts, expected {size}'
    digest = subprocess.run(['sha256sum', path], capture_output=True, text=True, check=True).stdout.split()[0]
    if digest != sha:
        os.remove(path)
        raise AssertionError(f'{name}: sha256 {digest}, expected {sha} (removed: run the cell again)')
    open(ok, 'w').write(sha)
    return f'{name}: {size/2**30:.2f} GiB in {(time.time() - t0)/60:.1f} min, sha256 checked'

t0 = time.time()
with ThreadPoolExecutor(DOWNLOADS) as ex:
    for line in ex.map(fetch, WANT):
        print(line, flush=True)
print(f'all {len(WANT)} files in {(time.time() - t0)/60:.1f} min')

DATA_FILES = [f'{LOCAL}/{p["file"]}' for p in MANIFEST['parts']]
DATA = ','.join(DATA_FILES)
SAMPLE = f'{LOCAL}/{HOLD["file"]}'               # the census reads its first 600k records
total = sum(os.path.getsize(f) for f in DATA_FILES) // 32
assert total == V10_RECORDS
sbs = sum(sb for _, sb, _ in PHASES)
print(f'\ntraining stream: {total/1e9:.3f}B records, {sbs} superbatches = {sbs*6104*16384/total:.2f} epochs')

In [ ]:
# --- stage helpers and census (RUN THIS ALSO WHEN RESTARTING) ---

def census(qbin):
    """Dead l1 neurons per output bucket: all-zero rows, and the activation census."""
    sh(f'{TRAINER}/target/release/sanity_deep {qbin} | grep "dead l1"')
    # l1census is the engine's own NNUE code, which reads the shipped layout only.
    if os.path.exists(L1CENSUS) and os.path.getsize(qbin) == SHIPPED_BYTES:
        sh(f'{L1CENSUS} {qbin} {SAMPLE} 0 600000')

def stage_cmd(end, start, resume_from):
    # STAGE_END and the recipe variables must sit right in front of cargo (after `env`):
    # `STAGE_END=40 cd dir && cargo ...` gives the variable to `cd` only.
    args = f'{DATA} {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return f'{CARGO} {ENV_PREFIX} STAGE_END={end} cargo run -r --bin trainer_deep --features cuda -- {args}'

def check_output(out, first_stage):
    assert TRAINER_LINE in out, f'the trainer did not print "{TRAINER_LINE}": wrong build'
    recipe = re.search(r'^recipe: .*$', out, re.M)
    assert recipe, 'no recipe line: wrong branch'
    for v in ENV.values():
        assert v in recipe.group(0), f'recipe line misses {v}: {recipe.group(0)}'
    assert ('revive true' in recipe.group(0)) == ('HYDRAY_REVIVE' in ENV)
    assert ('qat true' in recipe.group(0)) == ('HYDRAY_QAT' in ENV)
    assert f'l2 {L2}' in recipe.group(0), 'the trainer did not take HYDRAY_L2'
    if first_stage and 'HYDRAY_INIT_NET' in ENV:
        assert out.count('init check ') == 5, 'the start check did not run'

def check_checkpoint(end):
    q = f'{TRAINER}/checkpoints/{NET_ID}-{end}/quantised.bin'
    assert os.path.exists(q), f'missing {q}'
    assert os.path.getsize(q) == NET_FILE_BYTES, f'{q}: {os.path.getsize(q)} bytes, not the 8-bucket 16 -> 32 -> 1 layout'
    return q

def save_to_drive(end):
    # Copies the checkpoint to Drive and CHECKS it arrived: the Drive mount writes
    # through a cache, so an unfinished upload would pass for a finished one.
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'{DRIVE}/{NET_ID}-{end}'
    sh(f'rm -rf "{dst}" && cp -r {ck} {DRIVE}/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst) and size(dst) == size(ck), f'incomplete copy on Drive: {dst}'
    print(f'superbatch {end} on Drive ({size(dst)/2**20:.0f} MiB, checked)', flush=True)

def run_stages(done=0):
    assert done % STAGE == 0, f'{done} is not a stage boundary'
    prev = done if done else None
    for k in range(done // STAGE, TOTAL_SB // STAGE):
        end, start = (k + 1) * STAGE, k * STAGE + 1
        print(f'\n===== {PHASE}, stage {k+1}/{TOTAL_SB//STAGE}: superbatches {start}-{end} =====', flush=True)
        out = sh(stage_cmd(end, start, prev))
        check_output(out, prev is None)
        census(check_checkpoint(end))
        save_to_drive(end)
        prev = end

def finish():
    """Final checks of the current phase, and its net on Drive (the next phase starts from it)."""
    final = check_checkpoint(TOTAL_SB)
    sh(f'{TRAINER}/target/release/sanity_deep {final}')
    census(final)
    sh(f'cp {final} "{DRIVE}/{NET_ID}.nnue" && cmp {final} "{DRIVE}/{NET_ID}.nnue"')
    print(f'\nnet on Drive: {DRIVE}/{NET_ID}.nnue')
    print('Sanity: startpos small, the mirrored pair (wN e4 / bN e5) identical, a queen up ~ +1000 or more.')

def run_arm(first=0, done=0):
    """Runs the phases from `first` on; `done` superbatches of phase `first` are already trained."""
    for i in range(first, len(PHASES)):
        use_phase(i)
        if done == 0 and os.path.exists(f'{DRIVE}/{NET_ID}.nnue'):
            print(f'{NET_ID}.nnue already on Drive: phase {PHASE} skipped', flush=True)
            continue
        run_stages(done=done)
        finish()
        done = 0


In [ ]:
# --- training ---
# Do NOT run this cell when restarting: use the cell "RESTART" below.
census(INIT_NET)                                  # the shipped net on the v10 holdout, for comparison
run_arm()

In [ ]:
# --- the arm's nets on Drive ---
# Each phase is checked and copied to Drive by run_arm (finish), the last one included.
for i in range(len(PHASES)):
    use_phase(i)
    net = f'{DRIVE}/{NET_ID}.nnue'
    print(f'{net}: {"on Drive" if os.path.exists(net) else "MISSING"}')

In [ ]:
# --- RESTART (ONLY if the session died mid-run) ---
#   1. run the cells from "helper" to "stage helpers" (the data cell reuses the parts
#      already on local disk, or decompresses them again in a new session);
#   2. do NOT run the training cell;
#   3. set RESTART = True, PHASE_ID = the phase that was running and DONE = its last
#      superbatch saved on Drive. The phases after it then run as usual.
# With RESTART = False this cell does nothing, so "Run all" is safe.
RESTART  = False
PHASE_ID = 'antidead'
DONE     = 20

if not RESTART:
    print('restart off (RESTART = False): nothing to do')
else:
    first = [p[0] for p in PHASES].index(PHASE_ID)
    use_phase(first)
    ck = f'{DRIVE}/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint not found on Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r "{ck}" {TRAINER}/checkpoints/')
    check_checkpoint(DONE)
    run_arm(first, DONE)

## After the run

The arm's net is the last phase's: `hydray-v10-l2x32-antidead-160sb.nnue` (`pre`),
`hydray-v10-l2x32-ft-shipped-qat-40sb.nnue` (`ft`), `hydray-v10-l2x32-ft-antidead-qat-40sb.nnue`
(`both`).

**The engine cannot load this layout yet.** `nnue_deep.cpp` and the tools built on it
(`reorder`, `l1census`, `holdoutloss`, `nnue-selftest`) read the 16 -> 1 tail only. Before
any check or SPRT the engine needs the second layer: loader, forward (float first, as
the file; int8 later as a speed step), and the agreement test with `sanity_deep.rs`, the
reference reader of this layout. On the laptop, the trainer side alone:

    cd nnue/trainer && cargo run -r --bin sanity_deep -- <net>.nnue

prints the layout (second hidden layer 32), the dead l1 rows and the sample evals.

Then, with that engine build, as for every net: reorder for sparsity, `nnue-selftest`,
census and holdout loss against `nnue/net/hydray.nnue`, and the SPRT against the shipped
net, both nets through `EvalFile`, 4+0.04 and 10+0.1, Threads=1, [0, 5], 8000-game cap,
one SPRT at a time. A new net is a new engine version. Do not SPRT a checkpoint taken
before the end of its LR schedule.
